<a href="https://colab.research.google.com/github/Airdox/Cloning-App/blob/main/h.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
config_content = f"""
model:
  name: F5TTS
  backbone: DiT
  tokenizer: pinyin
  tokenizer_path: "/content/F5-TTS/data/my_voice_pinyin/vocab.txt"
  arch:
    dim: 1024
    depth: 22
    heads: 16
    dim_head: 64
    ff_mult: 2
  mel_spec:
    target_sample_rate: 24000
    n_mel_channels: 192
    hop_length: 256
    win_length: 1024
    n_fft: 1024
    mel_spec_type: vocos
  vocoder:
    is_local: false
    local_path: null

datasets:
  name: "my_voice"
  path: "/content/F5-TTS/data/my_voice_pinyin"
  batch_size_per_gpu: 2
  batch_size_type: sample
  max_samples: 64
  num_workers: 2

optim:
  epochs: 100
  learning_rate: 0.00005
  num_warmup_updates: 1000
  grad_accumulation_steps: 1
  max_grad_norm: 2.0
  bnb_optimizer: False

ckpts:
  logger: tensorboard
  log_samples: true
  save_per_updates: 500
  last_per_updates: 500
  keep_last_n_checkpoints: 5
  wandb_project: F5-TTS-Finetune
  save_dir: /content/f5_finetune_output/ckpts

train:
  seed: 42
  mixed_precision: bf16
  save_dtype: bfloat16
  num_workers: 2
  log_samples: true
  ckpt_resume: null

exp:
  name: my_voice_finetune
  output_dir: /content/f5_finetune_output
  model_path: null
  log_interval: 50
"""

with open('/content/f5_finetune_config.yaml', 'w') as f:
    f.write(config_content.strip())

print(f'✅ Konfiguration finalisiert.')

In [ ]:
import os, shutil

# Erstelle das Verzeichnis, in dem das Skript die Vokabular-Datei erwartet
expected_vocab_dir = '/content/f5_train_dataset/processed_pinyin'
os.makedirs(expected_vocab_dir, exist_ok=True)

# Kopiere die verifizierte vocab.txt dorthin
source_vocab = '/content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt'
dest_vocab = os.path.join(expected_vocab_dir, 'vocab.txt')

shutil.copy2(source_vocab, dest_vocab)

print(f'✅ vocab.txt kopiert nach: {dest_vocab}')

In [ ]:
!accelerate launch --num_processes 1 --num_machines 1 --dynamo_backend no --mixed_precision bf16 src/f5_tts/train/train.py --config-path /content --config-name f5_finetune_config

# 🎙️ Voice Cloning mit F5-TTS

**Zero-Shot Stimme klonen + Optional Fine-Tuning**

F5-TTS: MIT-Lizenz, aktuell gewartet, hochqualitativ, mehrsprachig (inkl. Deutsch).

⚠️ **WICHTIG:** `Laufzeit` → `Laufzeittyp ändern` → **T4 GPU** auswählen!

⏱️ Gesamtdauer: ~3–5 Minuten (Zero-Shot) | ~30–60 Min (mit Fine-Tuning)

In [ ]:
!nvidia-smi

## 1. Installation (sauber, keine Warnings)

In [8]:
import subprocess, sys, importlib, os

def install_and_verify(package, import_name=None, extra_args=None):
    """Installiert ein Paket und prüft den Import."""
    imp = import_name or package.split('[')[0].replace('-', '_')
    try:
        importlib.import_module(imp)
        print(f'  ✅ {package} – bereits installiert')
        return True
    except ImportError:
        pass
    cmd = [sys.executable, '-m', 'pip', 'install', '-q']
    if extra_args:
        cmd.extend(extra_args)
    cmd.append(package)
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        cmd2 = [sys.executable, '-m', 'pip', 'install'] + (extra_args or []) + [package]
        r2 = subprocess.run(cmd2, capture_output=True, text=True)
        if r2.returncode != 0:
            print(f'  ❌ {package}: {r2.stderr[-400:]}')
            return False
    try:
        importlib.import_module(imp)
        print(f'  ✅ {package} – erfolgreich installiert')
        return True
    except ImportError:
        print(f'  ⚠️ {package} installiert, aber Import schlägt fehl.')
        return False

print('📦 Installiere notwendige Pakete für das Training...\n')

# Install Hydra and training dependencies
install_and_verify('hydra-core', 'hydra')
install_and_verify('accelerate')
install_and_verify('transformers')
install_and_verify('vocos')

# Ensure local F5-TTS package is correctly registered
f5_tts_repo_path = '/content/F5-TTS'
if os.path.exists(f5_tts_repo_path):
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', f5_tts_repo_path], capture_output=True)
    print('  ✅ f5-tts – editable install aus lokalem Repo erneuert.')

print('\n✅ Vorbereitung abgeschlossen!')

📦 Installiere notwendige Pakete für das Training...

  ✅ hydra-core – erfolgreich installiert
  ✅ accelerate – bereits installiert
  ✅ transformers – bereits installiert
  ✅ vocos – erfolgreich installiert
  ✅ f5-tts – editable install aus lokalem Repo erneuert.

✅ Vorbereitung abgeschlossen!


In [ ]:
import torch
from f5_tts.api import F5TTS

print(f'PyTorch:  {torch.__version__}')
print(f'CUDA:     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU:      {torch.cuda.get_device_name(0)}')
    print(f'VRAM:     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'F5-TTS:   geladen ✅')

In [ ]:
# Modell laden (erster Start lädt ~1.5 GB)
print('⏳ Lade F5-TTS Modell...')
f5tts = F5TTS(device='cuda:0' if torch.cuda.is_available() else 'cpu')
print('✅ Modell geladen!')

---
## 4. Fine-Tuning (Optional – für höchste Qualität)

> **Brauchst du das?** Zero-Shot F5-TTS ist bereits sehr gut. Fine-Tuning hilft bei:
> - Sehr spezifischen Sprechgewohnheiten
> - Professionellen Anwendungen
> - Extrem natürlicher Klangfarbe

### Voraussetzungen:
- **10–30 Minuten** sauberes Audio deiner Stimme
- Klare Aussprache, gleichmäßige Lautstärke
- **~30–60 Minuten** Trainingszeit auf T4 GPU

In [9]:
# Trainingsmaterial hochladen (10–30 Minuten Audio)
# Du kannst mehrere Dateien auf einmal auswählen!
import os, shutil
from google.colab import files as colab_files

train_dir = '/content/train_audio'
os.makedirs(train_dir, exist_ok=True)

print('📂 Lade jetzt dein Trainingsmaterial hoch (10–30 Min.):')
print('   Mehrere Dateien sind erlaubt!\n')

# Fix: Nutze colab_files statt files, um Namenskonflikte zu vermeiden
uploaded_train = colab_files.upload()

train_files = []
for fname, data in uploaded_train.items():
    dest = os.path.join(train_dir, fname)
    with open(dest, 'wb') as f:
        f.write(data)
    train_files.append(dest)
    print(f'  → {fname} ({len(data)/1024/1024:.1f} MB)')

print(f'\n✅ {len(train_files)} Datei(en) gespeichert.')

📂 Lade jetzt dein Trainingsmaterial hoch (10–30 Min.):
   Mehrere Dateien sind erlaubt!



Saving WinRAR-ZIP-Archiv (neu).zip to WinRAR-ZIP-Archiv (neu).zip
  → WinRAR-ZIP-Archiv (neu).zip (0.9 MB)

✅ 1 Datei(en) gespeichert.


In [11]:
import os, shutil
if os.path.exists('/content/F5-TTS'):
    shutil.rmtree('/content/F5-TTS')

print('⏳ Klone F5-TTS Repository erneut...')
!git clone https://github.com/SWivid/F5-TTS.git /content/F5-TTS
print('✅ Repository wiederhergestellt.')

⏳ Klone F5-TTS Repository erneut...
Cloning into '/content/F5-TTS'...
remote: Enumerating objects: 4023, done.
remote: Counting objects: 100% (1734/1734), done.
remote: Compressing objects: 100% (339/339), done.
remote: Total 4023 (delta 1529), reused 1395 (delta 1395), pack-reused 2289 (from 2)
Receiving objects: 100% (4023/4023), 2.36 MiB | 2.03 MiB/s, done.
Resolving deltas: 100% (2472/2472), done.
✅ Repository wiederhergestellt.


In [19]:
!pip install -q faster-whisper pydub

from pydub import AudioSegment
from faster_whisper import WhisperModel
import glob, json, os, zipfile

# 0) ZIP-Dateien entpacken
train_dir = '/content/train_audio'
os.makedirs(train_dir, exist_ok=True)
for f in glob.glob(os.path.join(train_dir, '*.zip')):
    print(f'📦 Entpacke {os.path.basename(f)}...')
    with zipfile.ZipFile(f, 'r') as zip_ref:
        zip_ref.extractall(train_dir)

supported_ext = ('.wav', '.mp3', '.m4a', '.ogg', '.opus', '.flac')
actual_train_files = [os.path.join(train_dir, f) for f in os.listdir(train_dir) if f.lower().endswith(supported_ext)]

SEGMENT_SEC = 8
MIN_SEC = 3
SAMPLE_RATE = 24000

print('🔗 Verarbeite Audiodateien...')
combined = AudioSegment.empty()
for f in sorted(actual_train_files):
    try:
        audio = AudioSegment.from_file(f).set_frame_rate(SAMPLE_RATE).set_channels(1)
        combined += audio
    except Exception as e: print(f'  ❌ {os.path.basename(f)}: {e}')

if len(combined) > 0:
    print(f'⏱️ Gesamtlänge: {len(combined)/1000:.1f}s')
    segments = []
    for i in range(0, len(combined), (SEGMENT_SEC-1)*1000):
        seg = combined[i:i+SEGMENT_SEC*1000]
        if len(seg) >= MIN_SEC*1000: segments.append(seg)

    print('\n🗣️ Lade Whisper & Transkribiere...')
    whisper = WhisperModel('small', device='cpu', compute_type='int8')
    f5_dataset = '/content/f5_train_dataset'
    wavs_dir = os.path.join(f5_dataset, 'wavs')
    os.makedirs(wavs_dir, exist_ok=True)

    metadata_lines = []
    for i, seg in enumerate(segments):
        wav_path = os.path.join(wavs_dir, f'{i:05d}.wav')
        seg.export(wav_path, format='wav')
        res, _ = whisper.transcribe(wav_path)
        text = ' '.join([s.text.strip() for s in res]).strip()
        metadata_lines.append(f'{os.path.abspath(wav_path)}|{text}')

    with open(os.path.join(f5_dataset, 'metadata.csv'), 'w', encoding='utf-8') as f:
        f.write('audio_file|text\n')
        for line in metadata_lines: f.write(f'{line}\n')
    print(f'✅ Dataset bereit: {len(metadata_lines)} Samples.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.5/39.5 MB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 71.6 MB/s eta 0:00:00
📦 Entpacke WinRAR-ZIP-Archiv (neu).zip...
🔗 Verarbeite Audiodateien...
⏱️ Gesamtlänge: 444.1s

🗣️ Lade Whisper & Transkribiere...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


✅ Dataset bereit: 64 Samples.


In [20]:
print('⏳ Extrahiere Features...')
import os
repo_src = '/content/F5-TTS/src'
script_path = '/content/F5-TTS/src/f5_tts/train/datasets/prepare_csv_wavs.py'
csv_path = '/content/f5_train_dataset/metadata.csv'
out_dir = '/content/f5_train_dataset/processed'

os.makedirs(out_dir, exist_ok=True)
if os.path.exists(script_path):
    !PYTHONPATH={repo_src} python3 {script_path} {csv_path} {out_dir}
    print('\n✅ Features bereit!')

⏳ Extrahiere Features...

Processing 64 audio files using 1 workers...
Processing chunk 1/1: 100% 64/64 [00:00<00:00, 3030.36it/s]
Converting texts to pinyin: 100% 1/1 [00:00<00:00, 146.06it/s]

Saving to /content/f5_train_dataset/processed ...
Writing to raw.arrow ...: 100% 64/64 [00:00<00:00, 653127.63it/s]

For processed, sample count: 64
For processed, vocab size is: 69
For processed, total 0.14 hours

✅ Features bereit!


### 4b. Features extrahieren & Training starten

Dies extrahiert zuerst Audio-Features, dann trainiert das Modell auf deiner Stimme.

⏱️ **Geschätzte Dauer:** 30–60 Minuten auf T4 GPU

In [ ]:
config_content = """
datasets:
  name: my_voice
  batch_size_per_gpu: 8000
  batch_size_type: frame
  max_samples: 64
  num_workers: 2
  path: /content/F5-TTS/data/my_voice_pinyin

optim:
  epochs: 100
  learning_rate: 0.00005
  num_warmup_updates: 0
  grad_accumulation_steps: 4
  max_grad_norm: 1.0
  bnb_optimizer: False

model:
  name: F5TTS_Custom
  tokenizer: pinyin
  tokenizer_path: /content/F5-TTS/data/my_voice_pinyin/vocab.txt
  backbone: DiT
  arch:
    dim: 1024
    depth: 22
    heads: 16
    ff_mult: 2
    text_dim: 512
    text_mask_padding: False
    conv_layers: 4
    pe_attn_head: 1
    attn_backend: torch
    attn_mask_enabled: False
    checkpoint_activations: True
  mel_spec:
    target_sample_rate: 24000
    n_mel_channels: 100
    hop_length: 256
    win_length: 1024
    n_fft: 1024
    mel_spec_type: vocos
  vocoder:
    is_local: False
    local_path: null

ckpts:
  logger: tensorboard
  wandb_project: F5-TTS-Finetune
  wandb_run_name: custom_run
  wandb_resume_id: null
  log_samples: True
  save_per_updates: 500
  keep_last_n_checkpoints: 5
  last_per_updates: 500
  save_dir: /content/f5_finetune_output/ckpts
"""

config_path = '/content/f5_finetune_config.yaml'
with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f'✅ Scheduler configuration adjusted to prevent ZeroDivisionError.')

In [ ]:
import os
# Read the official config to verify the structure
base_config_path = '/content/F5-TTS/src/f5_tts/configs/F5TTS_Base.yaml'
if os.path.exists(base_config_path):
    print(f'--- Content of {base_config_path} ---')
    with open(base_config_path, 'r') as f:
        print(f.read())
else:
    print('❌ Base config not found in the expected path.')

In [ ]:
import os
repo_src = '/content/F5-TTS/src'

print('🚀 Starte Training...')

!export HYDRA_FULL_ERROR=1 && PYTHONPATH={repo_src} accelerate launch \
    --mixed_precision=fp16 \
    -m f5_tts.train.train \
    --config-path /content \
    --config-name f5_finetune_config

In [ ]:
import os
import json

target_dir = '/content/F5-TTS/data/my_voice_pinyin'
# Ensure the target directory exists
os.makedirs(target_dir, exist_ok=True)
duration_json_path = os.path.join(target_dir, 'duration.json')

# The script expects a dictionary with a 'duration' key containing the list
dummy_durations = [8.0] * 16
data_to_save = {'duration': dummy_durations}

with open(duration_json_path, 'w', encoding='utf-8') as f:
    json.dump(data_to_save, f)

print(f"✅ Successfully corrected {duration_json_path} format (dict with 'duration' key).")

### 4c. Fine-getuntes Modell testen

In [2]:
import os
import glob

# Suche global nach .pt Dateien im f5_finetune_output Ordner
print('🔍 Suche nach Checkpoints...')
search_path = '/content/f5_finetune_output/**/*.pt'
ckpts = sorted(glob.glob(search_path, recursive=True))

if not ckpts:
    # Zweiter Versuch: Suche überall in /content
    ckpts = sorted(glob.glob('/content/**/*.pt', recursive=True))

print('📦 Gefundene Checkpoints:')
for c in ckpts:
    print(f'   {c} ({os.path.getsize(c)/1024/1024:.1f} MB)')

if ckpts:
    best_ckpt = ckpts[-1]
    print(f'\n✅ Nutze: {best_ckpt}')
else:
    print('\n⚠️ Keine Checkpoints gefunden. Bitte prüfe, ob der Ordner /content/f5_finetune_output existiert.')
    !ls -R /content/f5_finetune_output

🔍 Suche nach Checkpoints...
📦 Gefundene Checkpoints:
   /content/F5-TTS/content/f5_finetune_output/ckpts/model_last.pt (3205.9 MB)

✅ Nutze: /content/F5-TTS/content/f5_finetune_output/ckpts/model_last.pt


In [5]:
import torch
from f5_tts.api import F5TTS
from IPython.display import Audio, display
import os

# Pfad zum gefundenen Checkpoint
best_ckpt = '/content/F5-TTS/content/f5_finetune_output/ckpts/model_last.pt'

# Referenz-Audio definieren
train_audio_dir = '/content/train_audio'
ref_files = [os.path.join(train_audio_dir, f) for f in os.listdir(train_audio_dir) if f.endswith(('.opus', '.wav', '.mp3'))]
ref_audio = ref_files[0] if ref_files else None
ref_text = ""

if ref_audio and os.path.exists(best_ckpt):
    print(f'⏳ Lade fine-tuned Modell von: {best_ckpt}')
    # Korrektur: In der aktuellen API wird der Pfad oft als 'model_type' (für das Schema)
    # und 'ckpt_file' übergeben, oder man nutzt die Standard-Initialisierung mit dem Pfad.
    try:
        f5tts_ft = F5TTS(model_type="F5TTS", ckpt_file=best_ckpt, device='cuda:0' if torch.cuda.is_available() else 'cpu')
    except TypeError:
        # Fallback falls die API-Struktur variiert
        f5tts_ft = F5TTS(ckpt_file=best_ckpt, device='cuda:0' if torch.cuda.is_available() else 'cpu')

    test_text = 'Dies ist ein Test mit meinem neu trainierten Modell. Die Qualität sollte nun deutlich besser sein.'

    print('\n🔊 Generiere: Original (Zero-Shot)...')
    wav_orig, sr_orig, _ = f5tts.infer(ref_file=ref_audio, ref_text=ref_text, gen_text=test_text)
    display(Audio(wav_orig, rate=sr_orig, label='Original (Zero-Shot)'))

    print('\n🔊 Generiere: Fine-Tuned...')
    wav_ft, sr_ft, _ = f5tts_ft.infer(ref_file=ref_audio, ref_text=ref_text, gen_text=test_text)
    display(Audio(wav_ft, rate=sr_ft, label='Fine-Tuned'))
else:
    print('❌ Referenz-Audio oder Checkpoint nicht gefunden.')

⏳ Lade fine-tuned Modell von: /content/F5-TTS/content/f5_finetune_output/ckpts/model_last.pt
Download Vocos from huggingface charactr/vocos-mel-24khz


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(



vocab :  /content/F5-TTS/src/f5_tts/infer/examples/vocab.txt
token :  custom
model :  /content/F5-TTS/content/f5_finetune_output/ckpts/model_last.pt 



RuntimeError: PytorchStreamReader failed locating file data/1062: file not found. This is an internal miniz error. If you are seeing this error, there is a high likelihood that your checkpoint file is corrupted. This can happen if the checkpoint was not saved properly, was transferred incorrectly, or the file was modified after saving.

## 5. Alle Ergebnisse herunterladen

In [ ]:
import zipfile
from google.colab import files as colab_files

# Alle WAV-Dateien + Modell in ZIP packen
with zipfile.ZipFile('voice_clone_ergebnis.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    # Audio-Ergebnisse
    for wav in sorted(glob.glob('/content/output_*.wav')) + sorted(glob.glob('/content/vergleich_*.wav')):
        zf.write(wav, f'audio/{os.path.basename(wav)}')
        print(f'  📦 audio/{os.path.basename(wav)}')

    # Fine-tuned Modell
    if 'best_ckpt' in dir() and os.path.exists(best_ckpt):
        zf.write(best_ckpt, f'model/{os.path.basename(best_ckpt)}')
        print(f'  📦 model/{os.path.basename(best_ckpt)}')

    # Referenz-Audio
    zf.write(ref_audio, f'original/{ref_audio}')
    print(f'  📦 original/{ref_audio}')

size_mb = os.path.getsize('voice_clone_ergebnis.zip') / 1024 / 1024
print(f'\n📦 ZIP-Größe: {size_mb:.1f} MB')
colab_files.download('voice_clone_ergebnis.zip')

In [ ]:
import os
import wave

folder = "/content/F5-TTS/data/my_voice_pinyin/wavs"

count = 0
duration = 0

for file in os.listdir(folder):
    if file.endswith(".wav"):
        count += 1
        with wave.open(os.path.join(folder, file), "r") as w:
            duration += w.getnframes() / w.getframerate()

print("Dateien:", count)
print("Dauer (Minuten):", round(duration / 60, 2))


In [14]:
config_content = r'''
datasets:
  name: my_voice
  batch_size_per_gpu: 2
  batch_size_type: sample
  max_samples: 64
  num_workers: 2
  path: /content/f5_train_dataset/processed

optim:
  epochs: 100
  learning_rate: 0.00005
  num_warmup_updates: 100
  grad_accumulation_steps: 4
  max_grad_norm: 1.0
  bnb_optimizer: false

model:
  name: F5TTS
  tokenizer: pinyin
  tokenizer_path: /content/F5-TTS/data/my_voice_pinyin/vocab.txt
  backbone: DiT
  arch:
    dim: 1024
    depth: 22
    heads: 16
    dim_head: 64
    ff_mult: 2
  mel_spec:
    target_sample_rate: 24000
    n_mel_channels: 192
    hop_length: 256
    win_length: 1024
    n_fft: 1024
    mel_spec_type: vocos
  vocoder:
    is_local: false
    local_path: null

ckpts:
  logger: tensorboard
  wandb_project: F5-TTS-Finetune
  log_samples: true
  save_per_updates: 500
  keep_last_n_checkpoints: 5
  last_per_updates: 500
  save_dir: /content/f5_finetune_output/ckpts
'''

config_path = '/content/f5_finetune_config.yaml'
with open(config_path, 'w') as f:
    f.write(config_content.strip())

print(f"✅ YAML erstellt unter {config_path}")

✅ YAML erstellt unter /content/f5_finetune_config.yaml


In [ ]:
import os

# Pfad zum f5_tts Paket innerhalb des geklonten Repositories
f5_tts_package_path = '/content/F5-TTS/src/f5_tts'

print(f"Inhalt von {f5_tts_package_path}:")
if os.path.exists(f5_tts_package_path):
    for item in os.listdir(f5_tts_package_path):
        item_path = os.path.join(f5_tts_package_path, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")
else:
    print(f"❌ Pfad nicht gefunden: {f5_tts_package_path}")

# Zusätzlich den Inhalt des 'text' Unterverzeichnisses prüfen
f5_tts_text_path = os.path.join(f5_tts_package_path, 'text')
print(f"\nInhalt von {f5_tts_text_path}:")
if os.path.exists(f5_tts_text_path):
    for item in os.listdir(f5_tts_text_path):
        item_path = os.path.join(f5_tts_text_path, item)
        if os.path.isdir(item_path):
            print(f"  [DIR] {item}")
        else:
            print(f"  [FILE] {item}")
else:
    print(f"❌ Pfad nicht gefunden: {f5_tts_text_path}")

In [17]:
import os, sys, pandas as pd

# Ensure the directory exists
vocab_dir = '/content/F5-TTS/data/my_voice_pinyin'
os.makedirs(vocab_dir, exist_ok=True)
vocab_path = os.path.join(vocab_dir, 'vocab.txt')

# Path to metadata
csv_path = '/content/f5_train_dataset/metadata.csv'

if os.path.exists(csv_path):
    # Add repo to path to use the tokenizer
    repo_src = '/content/F5-TTS/src'
    if repo_src not in sys.path: sys.path.insert(0, repo_src)
    from f5_tts.text.tokenizer import PinyinTokenizer

    print(f"⏳ Generiere {vocab_path}...")
    # Use sep='|' as previously defined in dataset creation
    df = pd.read_csv(csv_path, sep='|', header=0, names=['audio_file', 'text'])
    tokenizer = PinyinTokenizer()

    unique_chars = set()
    for text in df['text'].dropna():
        for char in tokenizer.tokenize(text):
            unique_chars.add(char)

    with open(vocab_path, 'w', encoding='utf-8') as f:
        for char in sorted(list(unique_chars)):
            f.write(f"{char}\n")

    print(f"✅ {vocab_path} generiert ({len(unique_chars)} Zeichen).")
else:
    print("❌ metadata.csv nicht gefunden. Bitte sicherstellen, dass die Audio-Verarbeitung abgeschlossen wurde.")

❌ metadata.csv nicht gefunden. Bitte sicherstellen, dass die Audio-Verarbeitung abgeschlossen wurde.


In [ ]:
import os

wavs = [f for f in os.listdir("/content/f5_train_dataset/wavs")
        if f.endswith(".wav")]

print("WAV-Dateien:", len(wavs))

In [ ]:
import os

for root, dirs, files in os.walk("/content/F5-TTS"):
    if "vocab.txt" in files:
        print(os.path.join(root, "vocab.txt"))

In [ ]:
# Den gefundenen Pfad einfügen
tokenizer_path = "/content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt"

# Die Config mit dem neuen Pfad aktualisieren
# Wir ersetzen 'tokenizer_path: null' durch den korrekten Pfad
config_content = config_content.replace("tokenizer_path: null", f"tokenizer_path: '{tokenizer_path}'")

with open(config_path, 'w') as f:
    f.write(config_content)

print(f"✅ Konfiguration wurde aktualisiert. Pfad: {tokenizer_path}")

In [ ]:
%cd /content/F5-TTS

In [7]:
import yaml
import shutil
import os

# Pfad zur originalen Beispiel-Config im Repo (falls vorhanden)
repo_config_path = "/content/F5-TTS/config/finetune_zh_en.yaml"
output_config_path = "/content/f5_finetune_config.yaml"

if os.path.exists(repo_config_path):
    # Lade die offizielle Vorlage
    with open(repo_config_path, 'r') as f:
        config = yaml.safe_load(f)

    # Pfade und spezifische Einstellungen anpassen
    config['datasets']['path'] = "/content/f5_train_dataset/processed"
    config['datasets']['name'] = "my_voice"
    config['model']['tokenizer_path'] = "/content/F5-TTS/data/my_voice_pinyin/vocab.txt"
    config['exp']['output_dir'] = "/content/f5_finetune_output"

    # Speichern
    with open(output_config_path, 'w') as f:
        yaml.dump(config, f)
    print("✅ Konfiguration wurde erfolgreich aus der offiziellen Vorlage geladen und angepasst.")
else:
    print("⚠️ Vorlage nicht gefunden. Bitte sicherstellen, dass das Repo korrekt unter /content/F5-TTS liegt.")

⚠️ Vorlage nicht gefunden. Bitte sicherstellen, dass das Repo korrekt unter /content/F5-TTS liegt.


In [3]:
%%bash
# Sicher in das Verzeichnis wechseln
cd /content/F5-TTS

# Training starten
accelerate launch --num_processes 1 --num_machines 1 \
    --dynamo_backend no --mixed_precision bf16 \
    src/f5_tts/train/train.py \
    --config-path /content \
    --config-name f5_finetune_config

bash: line 2: cd: /content/F5-TTS: No such file or directory
/usr/bin/python3: can't open file '/content/src/f5_tts/train/train.py': [Errno 2] No such file or directory
Traceback (most recent call last):
  File "/usr/local/bin/accelerate", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/accelerate_cli.py", line 50, in main
    args.func(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 1407, in launch_command
    simple_launcher(args)
  File "/usr/local/lib/python3.12/dist-packages/accelerate/commands/launch.py", line 993, in simple_launcher
    raise subprocess.CalledProcessError(returncode=process.returncode, cmd=cmd)
subprocess.CalledProcessError: Command '['/usr/bin/python3', 'src/f5_tts/train/train.py', '--config-path', '/content', '--config-name', 'f5_finetune_config']' returned non-zero exit status 2.


CalledProcessError: Command 'b'# Sicher in das Verzeichnis wechseln\ncd /content/F5-TTS\n\n# Training starten\naccelerate launch --num_processes 1 --num_machines 1 \\\n    --dynamo_backend no --mixed_precision bf16 \\\n    src/f5_tts/train/train.py \\\n    --config-path /content \\\n    --config-name f5_finetune_config\n'' returned non-zero exit status 1.

In [ ]:
!accelerate launch --num_processes 1 --num_machines 1 --dynamo_backend no --mixed_precision bf16 src/f5_tts/train/train.py --config-path /content --config-name f5_finetune_config

In [ ]:
import os, shutil

target_data_dir = '/content/F5-TTS/data/my_voice_pinyin'
os.makedirs(target_data_dir, exist_ok=True)

source_processed = '/content/f5_train_dataset/processed'
if os.path.exists(source_processed):
    for item in os.listdir(source_processed):
        s = os.path.join(source_processed, item)
        d = os.path.join(target_data_dir, item)
        if os.path.exists(d): os.remove(d)
        shutil.move(s, d)

source_vocab = '/content/F5-TTS/data/Emilia_ZH_EN_pinyin/vocab.txt'
dest_vocab = os.path.join(target_data_dir, 'vocab.txt')
shutil.copy2(source_vocab, dest_vocab)

print(f'✅ Daten verschoben nach: {target_data_dir}')

In [ ]:
import os, sys, shutil, pandas as pd

# 1. Pfade definieren
repo_root = '/content/F5-TTS'
repo_src = os.path.join(repo_root, 'src')
target_data_dir = '/content/F5-TTS/data/my_voice_pinyin'
vocab_path = os.path.join(target_data_dir, 'vocab.txt')
source_processed = '/content/f5_train_dataset/processed'
csv_path = '/content/f5_train_dataset/metadata.csv'

# 2. Daten an den richtigen Ort schieben
print("⏳ Bereite Daten für Training vor...")
os.makedirs(target_data_dir, exist_ok=True)

# Verschiebe die .arrow Dateien und andere Metadaten
if os.path.exists(source_processed):
    for item in os.listdir(source_processed):
        s = os.path.join(source_processed, item)
        d = os.path.join(target_data_dir, item)
        if os.path.exists(d):
            if os.path.isdir(d): shutil.rmtree(d)
            else: os.remove(d)
        shutil.copy2(s, d) if os.path.isfile(s) else shutil.copytree(s, d)
    print(f"✅ Trainingsdaten nach {target_data_dir} kopiert.")

# Sicherstellen, dass vocab.txt existiert
if not os.path.exists(vocab_path) and os.path.exists(csv_path):
    df = pd.read_csv(csv_path, sep='|')
    unique_chars = set(char for text in df['text'].dropna() for char in str(text))
    for token in ['<pad>', '<unk>', ' ']: unique_chars.add(token)
    with open(vocab_path, 'w', encoding='utf-8') as f:
        for char in sorted(list(unique_chars)):
            if char.strip() or char == ' ': f.write(f"{char}\n")
    print(f"✅ Vocab erstellt.")

# 3. Training Start
script_file = os.path.join(repo_src, 'f5_tts/train/train.py')
if os.path.exists(script_file):
    print('🚀 Starte Training...')
    !PYTHONPATH={repo_src} accelerate launch \
        --num_processes 1 --num_machines 1 --dynamo_backend no --mixed_precision bf16 \
        {script_file} --config-path /content --config-name f5_finetune_config
else:
    print("❌ Trainings-Skript nicht gefunden.")

⏳ Bereite Daten für Training vor...
✅ Trainingsdaten nach /content/F5-TTS/data/my_voice_pinyin kopiert.
🚀 Starte Training...
2026-06-25 15:43:03.940569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Using logger: tensorboard
Gradient accumulation checkpointing with per_updates now, old logic per_steps used with before f992c4e
Loading dataset ...
Download Vocos from huggingface charactr/vocos-mel-24khz
[2026-06-25 15:43:12,194][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/charactr/vocos-mel-24khz/resolve/main/config.yaml "HTTP/1.1 307 Temporary Redirect"
[2026-06-25 15:43:12,201][httpx][INFO] - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/charactr/vocos-mel-24khz/0feb3fdd929bcd6649e0e7c5a688cf7dd012ef21/config.y

---
### 💡 Tipps für beste Ergebnisse

| Thema | Empfehlung |
|-------|------------|
| **Referenz-Audio** | 5–30 Sek., sauber, keine Hintergrundgeräusche |
| **Transkription** | Text manuell eingeben statt Auto-Erkennung für höchste Qualität |
| **Training** | 10–30 Min. Audio, 50–100 Epochs, Batch Size 2 |
| **OOM Fehler** | `batch_size` in Config auf `1` reduzieren |
| **Sprache** | Deutsch wird automatisch erkannt |
| **Fine-Tuning nötig?** | Meistens reicht Zero-Shot bereits aus! |

### 🔧 Bei Problemen
- **Runtime neu starten** (`Laufzeit → Sitzung neu starten`) und ab Schritt 1 neu ausführen
- Bei Import-Fehler: `!pip install -U f5-tts` ausführen
- Bei OOM beim Training: Batch-Size reduzieren und Sitzung neu starten

### 📚 Ressourcen
- [F5-TTS GitHub](https://github.com/SWivid/F5-TTS)
- [F5-TTS Paper](https://arxiv.org/abs/2501.01458)
- Lizenz: **MIT** – auch kommerziell nutzbar!

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_loss():
    log_file = "/content/f5_finetune_output/logs/train.log" # Pfad zu deinem Log
    if not os.path.exists(log_file):
        print("Log-Datei noch nicht bereit. Warte auf den ersten Checkpoint...")
        return

    # Liest die Log-Daten (einfaches Beispiel)
    data = pd.read_csv(log_file)
    plt.figure(figsize=(10, 5))
    plt.plot(data['step'], data['loss'], label='Training Loss')
    plt.xlabel('Schritte')
    plt.ylabel('Loss')
    plt.title('Trainingsfortschritt')
    plt.legend()
    plt.show()

# Einmalig ausführen
plot_loss()